<a href="https://colab.research.google.com/github/KhalidMohd99/GeoSpectra-AI/blob/main/Mineral_Alteration_Analysis_using_ASTER_v01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ASTER-Based Mineral Alteration & Prospectivity Mapping Engine

**Eng. Khalid M.Ahmed Amlas** - *GIS Specialist*

This notebook demonstrates a workflow for **Gold Exploration using ASTER satellite imagery and Google Earth Engine (GEE)**.
It involves:

1.  Setting up the environment.
2.  Interactive Study Area Selection.
3.  Searching and Selecting ASTER Scenes.
4.  Applying Atmospheric Correction.
5.  Calculating various Band Ratios.
6.  Visualizing and Analyzing.
7.  Exporting.

### **1. Setup: Install Libraries and Initialize Google Earth Engine**

First, we need to install the `geemap` library, which simplifies interaction with Google Earth Engine in Colab. After installation, we import necessary libraries (`pandas`, `ee`, `geemap`).

**Google Earth Engine Authentication:** The `ee.Authenticate()` command will prompt you to log in to your Google account and grant GEE access. `ee.Initialize()` then sets up the GEE connection.

In [ ]:
# Install the necessary library
!pip install geemap

import pandas as pd
import ee
import geemap

# Authenticate and Initialize Earth Engine
# Follow the link that appears to log in
ee.Authenticate()
ee.Initialize(project='Project ID') # Replace with your Project ID if it asks

### **2. Create an Interactive Map for Study Area Selection**

To initializes an interactive map using `geemap`. You'll be able to pan, zoom, and most importantly, use the 'Marker' tool (found on the left sidebar of the map) to select your Area of Interest (AOI). After placing a point on the map, proceed to the next step.

In [ ]:
Map = geemap.Map()
Map.setCenter(33.5983, 18.7694, 6) # Default view over Sudan

print("INSTRUCTIONS:")
print("1. Wait for the map to load below.")
print("2. Use the 'Marker' tool (on the left toolbar) to click your study area.")
print("3. Once you have placed a point, run the NEXT cell.")

Map

INSTRUCTIONS:
1. Wait for the map to load below.
2. Use the 'Marker' tool (on the left toolbar) to click your study area.
3. Once you have placed a point, run the NEXT cell.


Map(center=[18.7694, 33.5983], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDa…

### **3. Get Coordinates and Search ASTER Collection**

Here, the notebook retrieves the coordinates of the point you marked on the map. If no point is marked, it uses a default location.

Then, it searches the ASTER L1T V003 image collection for scenes covering your chosen point. The results are displayed in a table, showing the date, cloud cover, and system ID for each available scene.

In [ ]:
# STEP 2: GET COORDINATES & SEARCH

# 1. Get the geometry from the map
# We check if the user drew something. If not, we use a default.
try:
    roi = Map.user_roi

    if roi is None:
        print("No point detected on the map! Using default coordinates (Sudan).")
        point = ee.Geometry.Point([33.98, 25.12])
    else:
        # Geemap returns a FeatureCollection or Geometry, we ensure it's a Point/Geometry
        point = roi if isinstance(roi, ee.Geometry) else roi.geometry()
        coords = point.centroid().coordinates().getInfo()
        print(f"Location detected: Longitude {coords[0]:.4f}, Latitude {coords[1]:.4f}")

except Exception as e:
    print(f"Error reading map: {e}")
    print("Using default coordinates.")
    point = ee.Geometry.Point([33.98, 25.12])

# 2. Search Collection
asterCollection = ee.ImageCollection("ASTER/AST_L1T_003") \
  .filterBounds(point) \
  .filterDate('2000-01-01', '2007-12-31') \
  .filter(ee.Filter.lt('CLOUDCOVER', 20)) \
  .sort('CLOUDCOVER')

count = asterCollection.size().getInfo()

if count == 0:
    print("No images found at this location with <20% cloud cover.")
else:
    print(f"\nFound {count} scenes. Generating list...")

    # Helper to format the list
    def get_meta(img):
        return ee.Feature(None, {
            'id': img.get('system:index'),
            'date': img.date().format('YYYY-MM-dd'),
            'cloud': img.get('CLOUDCOVER')
        })

    # Create DataFrame
    info_list = asterCollection.map(get_meta).getInfo()['features']
    df = pd.DataFrame([f['properties'] for f in info_list])

    # Reorder columns
    df = df[['date', 'cloud', 'id']]

    print("\nAVAILABLE SCENES:")
    print(df.to_string(index=True))
    print("\nKeep this list visible for the next step.")

Location detected: Longitude 32.7775, Latitude 21.1140

Found 15 scenes. Generating list...

AVAILABLE SCENES:
          date  cloud              id
0   2000-07-08      0  20000708084642
1   2000-07-08      0  20000708084651
2   2000-08-25      0  20000825084623
3   2000-08-25      0  20000825084632
4   2001-04-06      0  20010406084133
5   2001-04-06      0  20010406084142
6   2007-02-18      0  20070218083142
7   2007-02-18      0  20070218083151
8   2007-03-22      0  20070322083145
9   2007-03-22      0  20070322083154
10  2000-06-06      2  20000606084648
11  2000-06-06      2  20000606084657
12  2001-05-24      3  20010524084054
13  2001-05-24      3  20010524084103
14  2000-11-29      4  20001129084416

Keep this list visible for the next step.


### 4. Select and Visualize an ASTER Scene

From the list generated in the previous step, you will be prompted to enter the index number of the scene you wish to visualize. The selected ASTER image is then fetched and displayed on a new interactive map.

In [ ]:
# STEP 3: SELECT & VISUALIZE

# Interactive Input
selection = input(f"Enter the Index Number (0-{count-1}) of the scene to visualize: ")

try:
    # Get the ID from the DataFrame
    idx = int(selection)
    selected_id = df.iloc[idx]['id']
    print(f"\nProcessing Scene ID: {selected_id}...")

    # Fetch the image
    aster = ee.Image('ASTER/AST_L1T_003/' + selected_id)
    aoi = aster.geometry()

    # Create a Result Map
    ResultMap = geemap.Map()
    ResultMap.centerObject(aoi, 10)

    # Add Layers
    ResultMap.addLayer(point, {'color': 'yellow'}, 'Your Search Point')

    # Footprint
    empty = ee.Image().byte()
    outline = empty.paint(featureCollection=ee.FeatureCollection(aoi), color=1, width=3)
    ResultMap.addLayer(outline, {'palette': 'red'}, 'Scene Footprint')

    # Natural Color Image
    ResultMap.addLayer(aster, {'bands': ['B3N', 'B02', 'B01'], 'min': 0, 'max': 255}, 'ASTER Natural Color')

    print("Visualization ready below:")
    display(ResultMap)

except Exception as e:
    print(f"Error: {e}")
    print("Please make sure you entered a valid number from the list.")

Enter the Index Number (0-14) of the scene to visualize: 12

Processing Scene ID: 20010524084054...
Visualization ready below:


Map(center=[21.40221347910476, 32.72821244642158], controls=(WidgetControl(options=['position', 'transparent_b…

### **5. Atmospheric Correction**

This step applies a simple Dark Object Subtraction (DOS) atmospheric correction to the selected ASTER image. This method helps to reduce atmospheric effects by identifying the darkest pixel values in each band within the Area of Interest and subtracting them.

In [ ]:
def dos(img):
    # Select VNIR and SWIR bands
    ms = img.select('B0[1-9]', 'B3N')
    # Select TIR bands
    tir = img.select('B1[0-4]')

    # Find the darkest pixel value in the AOI for each band
    dark_object = ms.reduceRegion(
        reducer=ee.Reducer.min(),
        geometry=aoi,
        scale=30,
        maxPixels=1e13,
        bestEffort=True
    )

    # Subtract the dark object value from each band
    # We use a loop over band names
    def correct_band(band_name):
        do_val = ee.Number(dark_object.get(band_name))
        return ms.select([band_name]).subtract(do_val).max(0)

    bands = ms.bandNames()
    corrected_bands = bands.map(correct_band)

    # Recombine corrected VNIR/SWIR with original TIR
    return ee.ImageCollection(corrected_bands).toBands().rename(bands).addBands(tir)

# Apply correction
aster_cor = dos(aster)
print("Atmospheric correction complete.")

Atmospheric correction complete.


### **6. Calculate Gold Exploration Band Ratios**

This function (`calculateGoldRatios`) computes a set of spectral band ratios from the atmospherically corrected ASTER image. These ratios are empirically derived to highlight various alteration mineral assemblages commonly associated with gold mineralization, categorized into:

*   **Argillic & Advanced Argillic**
*   **Phyllic Zone**
*   **Propylitic Zone**
*   **Silicification**
*   **Iron Oxides**

In [ ]:
def calculateGoldRatios(img):
    # GROUP 1: ARGILLIC & ADVANCED ARGILLIC
    alunite_kao_pyro = img.expression('(b4 + b6) / b5',
        {'b4': img.select('B04'), 'b5': img.select('B05'), 'b6': img.select('B06')}).rename('Alunite_Kaolinite_Pyrophyllite')

    kaolinite = img.expression('b4 / b6',
        {'b4': img.select('B04'), 'b6': img.select('B06')}).rename('Kaolinite_Index')

    alunite = img.expression('b4 / b5',
        {'b4': img.select('B04'), 'b5': img.select('B05')}).rename('Alunite_Index')

    argillic_general = img.expression('(b5 + b7) / b6',
        {'b5': img.select('B05'), 'b6': img.select('B06'), 'b7': img.select('B07')}).rename('Argillic_General')

    # GROUP 2: PHYLLIC ZONE
    sericite_muscovite = img.expression('(b5 + b7) / b6',
        {'b5': img.select('B05'), 'b6': img.select('B06'), 'b7': img.select('B07')}).rename('Sericite_Muscovite')

    phyllic = img.expression('b5 / b6',
        {'b5': img.select('B05'), 'b6': img.select('B06')}).rename('Phyllic_Alteration')

    # GROUP 3: PROPYLITIC ZONE
    propylitic = img.expression('(b7 + b9) / b8',
        {'b7': img.select('B07'), 'b8': img.select('B08'), 'b9': img.select('B09')}).rename('Propylitic_Index')

    chlorite = img.expression('(b6 + b9) / b8',
        {'b6': img.select('B06'), 'b8': img.select('B08'), 'b9': img.select('B09')}).rename('Chlorite_Index')

    epidote_calcite = img.expression('b13 / b14',
        {'b13': img.select('B13'), 'b14': img.select('B14')}).rename('Epidote_Calcite_Carbonate')

    # GROUP 4: SILICIFICATION
    quartz_index = img.expression('b11 / (b10 + b12)',
        {'b10': img.select('B10'), 'b11': img.select('B11'), 'b12': img.select('B12')}).rename('Quartz_Index_QI')

    silica_index = img.expression('b13 / b10',
        {'b10': img.select('B10'), 'b13': img.select('B13')}).rename('Silica_Index')

    # GROUP 5: IRON OXIDES
    gossan = img.expression('b4 / b2',
        {'b4': img.select('B04'), 'b2': img.select('B02')}).rename('Gossan_Index')

    ferric_hematite = img.expression('b2 / b1',
        {'b1': img.select('B01'), 'b2': img.select('B02')}).rename('Ferric_Iron_Hematite')

    ferrous = img.expression('b5 / b3',
        {'b5': img.select('B05'), 'b3': img.select('B3N')}).rename('Ferrous_Iron')

    return ee.Image.cat([
        alunite_kao_pyro, kaolinite, alunite, argillic_general,
        sericite_muscovite, phyllic, propylitic, chlorite,
        epidote_calcite, quartz_index, silica_index,
        gossan, ferric_hematite, ferrous
    ])

goldIndices = calculateGoldRatios(aster_cor)
print('Indices Calculated.')

Indices Calculated.


### **7. Visualize Band Ratio Maps**

Each calculated band ratio is added as a separate layer to the interactive map.

In [ ]:
# Visualization Parameters (Grayscale)
ratioVis = {'min': 0.8, 'max': 1.5, 'palette': ['black', 'white']}
thermalVis = {'min': 0.4, 'max': 0.6, 'palette': ['black', 'white']}

# Get band names as a Python list
band_names = goldIndices.bandNames().getInfo()

# Loop through bands and add layers
for band in band_names:
    vis = ratioVis
    # Apply thermal visualization to TIR indices
    if 'Quartz' in band or 'Epidote' in band:
        vis = thermalVis

    Map.addLayer(goldIndices.select(band), vis, band, False)

# Display the map
Map

Map(bottom=115412.0, center=[21.40221347910467, 32.728212446421225], controls=(WidgetControl(options=['positio…

### **9. Export Results to Google Drive**

Finally, the composite image containing all the calculated mineral alteration (band ratios) is exported to your Google Drive as a GeoTIFF file.

In [ ]:
# 6. EXPORTS

sceneId = aster.get('system:index').getInfo()
description = 'ASTER_Gold_Indices_Composite_' + sceneId

task = ee.batch.Export.image.toDrive(
    image=goldIndices.toFloat(),
    description=description,
    scale=30,
    region=aoi,
    fileFormat='GeoTIFF',
    crs='EPSG:4326',
    folder='GEE_Gold_Exploration',
    maxPixels=1e13
)

task.start()
print(f"Export started: {description}")
print("Check the 'Tasks' tab in your Google Earth Engine Code Editor or query task status here.")

Export started: ASTER_Gold_Indices_Composite_20000606084546
Check the 'Tasks' tab in your Google Earth Engine Code Editor or query task status here.
